In [1]:
import numpy as np
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.metrics.pairwise import rbf_kernel
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Gene Exp & Drug response

In [2]:
drug_response = pd.read_csv("../ctrp_data/drugAct.csv", index_col=0)
gene_exp = pd.concat(
    [
        pd.read_csv("../ctrp_data/gene_exp_part1.csv.gz", index_col=0).T,
        pd.read_csv("../ctrp_data/gene_exp_part2.csv.gz", index_col=0).T,
    ],
    axis=1,
)
gene_exp.columns = list(gene_exp.columns)
cells = sorted(
    set(drug_response.columns)
    & set(gene_exp.index)
    & set(pd.read_csv("../ctrp_data/mut.csv", index_col=0).T.index)
)

In [3]:
# def get_smiles_from_compound_name(compound_name, max_retries=5, delay=5):
#     # First try to get SMILES from local file
#     try:
#         df = pd.read_csv(
#             "../Figs/nsc_cid_smiles_class_name.csv", usecols=["NAME", "SMILES"]
#         )
#         match = df[df["NAME"].str.lower() == compound_name.lower()]
#         if not match.empty:
#             return match.iloc[0]["SMILES"]
#     except Exception as e:
#         print(f"Error reading local SMILES data: {e}")

#     # If not found locally, try PubChem API
#     url = f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{compound_name}/property/CanonicalSMILES/JSON"

#     for attempt in range(max_retries):
#         try:
#             response = requests.get(url)
#             response.raise_for_status()
#             data = response.json()
#             smiles = (
#                 data.get("PropertyTable", {})
#                 .get("Properties", [{}])[0]
#                 .get("CanonicalSMILES")
#             )
#             return smiles
#         except requests.RequestException as e:
#             if response.status_code == 503 or "PUGREST.ServerBusy" in str(e):
#                 print(
#                     f"Server busy. Retrying in {delay} seconds... (Attempt {attempt + 1}/{max_retries})"
#                 )
#                 time.sleep(delay)
#                 delay *= 2  # Exponential backoff
#             else:
#                 print(f"Error retrieving SMILES for compound name {compound_name}: {e}")
#                 return None

#     print(f"Failed to retrieve SMILES after {max_retries} attempts")
#     return None


# def process_compound(compound_name):
#     return [get_smiles_from_compound_name(compound_name), compound_name]


# def get_smiles_parallel(compound_names, max_workers=10):
#     results = []
#     with ThreadPoolExecutor(max_workers=max_workers) as executor:
#         futures = [executor.submit(process_compound, name) for name in compound_names]
#         for future in tqdm(
#             as_completed(futures),
#             total=len(compound_names),
#             desc="Processing compounds",
#         ):
#             results.append(future.result())
#     return results

In [4]:
# compound_names = drug_response.index.tolist()
# SMILES = get_smiles_parallel(compound_names)

In [5]:
# SMILES = pd.DataFrame(SMILES)
# SMILES

In [6]:
# SMILES[SMILES[0].isna()].shape

In [7]:
# SMILES.columns = ["SMILES", "drugs"]
# SMILES.to_csv('ctrp_drug2smiles.csv')

In [8]:
SMILES = pd.read_csv("ctrp_drug2smiles.csv", index_col=0)
SMILES

,SMILES,drugs
0,C1CNP(=O)(OC1)N(CCCl)CCCl,cyclophosphamide
1,CC1=C2C(C(=O)C3(C(CC4C(C3C(C(C2(C)C)(CC1OC(=O)...,docetaxel
2,C1C2CC3CC1CC(C2)(C3)C4=C(C=CC(=C4)C5=C(C=C(C=C...,3-Cl-AHPC
3,CC(C)N(CCCNC(=O)NC1=CC=C(C=C1)C(C)(C)C)CC2C(C(...,BRD-A02303741
4,CC1=CC=CC=C1C(C(=O)NC2CCCCC2)N(C3=CC(=CC=C3)F)...,BRD-A05715709
...,...,...
476,COCCOC(=O)N1C2=C(C=C(C=C2)C#CCC(C(=O)OC)C(=O)O...,BRD1378
477,CN1CCN(CC1)C(=O)CC2CCC(C(O2)CO)NC(=O)C3=CC4=C(...,BRD-K96431673
478,C1OC2=C(O1)C=C3C(=C2)C(=O)C=C(N3)C4=CC=CC=C4F,CHM-1
479,C1=CC=C(C=C1)C#CS(=O)(=O)N,pifithrin-mu


In [9]:
gene_exp = gene_exp.loc[cells]
gene_exp

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2MP1,A4GALT,A4GNT,AA06,...,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1,ZZZ3
22Rv1,4.753564,4.766881,7.445061,4.824879,5.714456,3.696364,4.500844,4.386674,3.917018,4.372562,...,10.158100,11.869601,5.920653,5.419161,6.434936,4.582009,7.828927,6.269888,5.893025,8.193560
23132/87,4.069636,4.471635,6.272017,3.731857,4.850012,3.907985,4.563642,5.028797,4.158208,4.109767,...,9.601165,11.128318,6.809560,5.980287,6.442293,4.054162,7.730826,7.432269,6.046933,8.481589
253J,4.604071,4.348786,3.964781,4.306114,4.243581,3.755806,4.259876,4.688513,4.104812,4.210417,...,9.116314,10.600051,7.711232,5.488803,5.956357,4.842830,7.701066,7.865817,6.008365,7.998332
253J-BV,4.490440,4.347983,3.807591,4.224780,4.345274,3.846573,4.276782,4.720335,4.082143,4.107064,...,9.775212,11.342161,7.804917,5.511868,6.501790,6.228556,7.662302,7.758116,6.054273,8.247543
42-MG-BA,7.290132,5.591604,4.030574,4.113296,3.277104,3.985902,4.839848,4.578237,3.898615,4.481621,...,10.125317,11.147441,5.051197,4.604050,5.914544,4.521433,8.257692,8.182643,6.077138,8.575384
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YH-13,6.410184,5.131086,4.122665,4.300066,3.516374,3.784239,4.432248,4.430834,3.877915,4.360993,...,9.502173,9.821473,5.659853,5.029787,6.141774,4.535534,8.164728,9.785886,5.898753,8.321323
YKG1,5.356574,4.778006,3.886142,3.949283,3.516238,3.777864,4.642822,4.386165,4.006922,4.065240,...,9.676738,11.067116,5.847052,5.401979,6.354059,5.201391,8.482566,8.045826,6.167265,8.226797
ZR-75-1,6.694777,5.002649,3.754769,3.769402,5.618349,3.740150,4.627128,4.574043,4.416972,3.714532,...,9.556498,11.629407,5.532236,5.133633,6.447122,7.039051,7.594835,5.964219,5.743329,7.752353
ZR-75-30,6.141592,4.766307,3.820361,4.617683,4.341006,4.029516,4.879128,4.666993,4.387402,4.269439,...,9.011994,10.617176,6.090865,5.601251,6.689855,6.921443,8.466809,7.642621,5.871723,7.907904


In [10]:
drug_response = drug_response.loc[sorted(SMILES.drugs), cells]
drug_response

,22Rv1,23132/87,253J,253J-BV,42-MG-BA,5637,639-V,647-V,697,769-P,...,YAPC,YD-10B,YD-15,YD-38,YD-8,YH-13,YKG1,ZR-75-1,ZR-75-30,huH-1
3-Cl-AHPC,19.0910,18.9320,18.273,18.971,20.4299,19.9749,19.298,19.9938,22.4386,19.7583,...,17.143,19.4010,19.7659,17.4660,16.5100,17.4850,15.4160,15.077,17.581,17.585
A-804598,NaN,NaN,NaN,NaN,15.2030,NaN,NaN,14.0380,NaN,NaN,...,14.350,NaN,NaN,NaN,14.8020,14.6720,NaN,14.350,15.276,14.350
ABT-199,NaN,NaN,NaN,NaN,14.1780,NaN,NaN,14.2240,26.1600,NaN,...,13.621,NaN,NaN,NaN,14.8110,13.6960,14.2350,14.350,14.667,14.350
ABT-737,16.3850,14.7270,16.098,15.748,14.3700,16.6040,13.611,NaN,22.3410,15.9300,...,14.350,14.7120,14.6520,14.5440,NaN,NaN,14.5410,14.874,14.945,NaN
AC55649,15.5050,15.5710,14.746,14.545,NaN,16.0460,13.671,14.3500,14.8350,14.5990,...,14.350,15.5620,13.5630,14.3500,15.2060,14.8760,14.6590,13.984,14.590,13.323
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,15.6330,14.3500,16.006,14.350,14.6650,15.5820,14.350,18.4040,17.0810,14.1500,...,13.071,13.8230,15.0480,14.6730,15.5900,16.5620,14.7190,14.350,NaN,18.192
vincristine,24.4091,24.5087,21.470,20.708,21.1354,22.8022,19.180,23.7750,29.1400,18.9950,...,17.569,23.0486,23.0685,22.0919,19.9137,24.1153,22.3504,18.280,18.868,19.235
vorapaxar,16.0300,14.9120,15.518,14.950,16.0440,16.8840,13.141,14.3850,NaN,14.8340,...,14.501,17.2930,15.4920,14.5540,17.6930,14.8210,NaN,15.279,14.682,14.960
vorinostat,18.9320,17.7120,18.416,16.850,17.0310,18.5220,15.485,17.6630,19.2720,17.1160,...,15.922,NaN,17.1210,24.3559,16.9690,16.2760,16.5080,16.024,16.996,16.824


# Standardized along with drugs

In [11]:
drug_response = drug_response.apply(lambda x: (x - np.nanmean(x)) / np.nanstd(x))

In [12]:
df = pd.DataFrame()
for i in drug_response.columns:
    tmp = pd.DataFrame(drug_response[i]).reset_index().dropna()
    tmp["cell"] = [i] * len(tmp)
    tmp.columns = ["Drug", "Value", "Cell"]
    tmp = tmp[["Drug", "Cell", "Value"]]
    df = pd.concat([df, tmp])
df

,Drug,Cell,Value
0,3-Cl-AHPC,22Rv1,1.148373
3,ABT-737,22Rv1,-0.059864
4,AC55649,22Rv1,-0.452787
5,AGK-2,22Rv1,-1.087268
6,AM-580,22Rv1,-0.796148
...,...,...,...
455,vemurafenib,huH-1,1.175953
456,vincristine,huH-1,1.671517
457,vorapaxar,huH-1,-0.359676
458,vorinostat,huH-1,0.525971


# Zero shot prediction

In [13]:
unique_drugs = df["Drug"].unique()
unique_cells = df["Cell"].unique()

# Split drugs and cell lines into training, validation, and test sets
train_drugs, test_val_drugs = train_test_split(
    unique_drugs, test_size=0.5, random_state=42
)
val_drugs, test_drugs = train_test_split(test_val_drugs, test_size=0.5, random_state=42)

train_cells, test_val_cells = train_test_split(
    unique_cells, test_size=0.55, random_state=42
)
val_cells, test_cells = train_test_split(test_val_cells, test_size=0.5, random_state=42)

# Split the dataset
train_df = df[(df["Drug"].isin(train_drugs)) & (df["Cell"].isin(train_cells))]
val_df = df[(df["Drug"].isin(val_drugs)) & (df["Cell"].isin(val_cells))]
test_df = df[(df["Drug"].isin(test_drugs)) & (df["Cell"].isin(test_cells))]


# Function to balance label distribution
def balance_labels(df, threshold=0.5):
    positive = df[df["Value"] > 0]
    negative = df[df["Value"] <= 0]
    min_count = min(len(positive), len(negative))
    balanced_positive = positive.sample(min_count, random_state=42)
    balanced_negative = negative.sample(min_count, random_state=42)
    return pd.concat([balanced_positive, balanced_negative])


# Balance label distribution across all sets
train_df = balance_labels(train_df)
val_df = balance_labels(val_df)
test_df = balance_labels(test_df)

# Separate features and labels
X_train = train_df[["Drug", "Cell"]]
y_train = np.array(train_df["Value"] > 0, dtype=float)

X_val = val_df[["Drug", "Cell"]]
y_val = np.array(val_df["Value"] > 0, dtype=float)

X_test = test_df[["Drug", "Cell"]]
y_test = np.array(test_df["Value"] > 0, dtype=float)

# Calculate total samples
total_samples = len(X_train) + len(X_val) + len(X_test)


# Function to calculate and format label ratios
def get_label_ratio(y):
    unique, counts = np.unique(y, return_counts=True)
    total = len(y)
    ratio_str = ", ".join(
        [f"Label {label}: {count/total:.2%}" for label, count in zip(unique, counts)]
    )
    return ratio_str


# Display results with percentages and label ratios
print("Train:")
print(X_train.shape, y_train.shape)
print(f"Percentage: {len(X_train)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_train)}")

print("\nValidation:")
print(X_val.shape, y_val.shape)
print(f"Percentage: {len(X_val)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_val)}")

print("\nTest:")
print(X_test.shape, y_test.shape)
print(f"Percentage: {len(X_test)/total_samples:.2%}")
print(f"Label Ratio: {get_label_ratio(y_test)}")

print(f"\nTotal samples: {total_samples}")
print(
    f"Ratio (Train:Validation:Test): {len(X_train):.0f}:{len(X_val):.0f}:{len(X_test):.0f}"
)
print(
    f"Overall Label Ratio: {get_label_ratio(np.concatenate([y_train, y_val, y_test]))}"
)

# X_train.to_csv("../nci_data/train.csv", index=False)
# X_test.to_csv("../nci_data/test.csv", index=False)
# X_val.to_csv("../nci_data/val.csv", index=False)

# np.save("../nci_data/train_labels.npy", y_train)
# np.save("../nci_data/test_labels.npy", y_test)
# np.save("../nci_data/val_labels.npy", y_val)

Train:
(51386, 2) (51386,)
Percentage: 59.86%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Validation:
(16596, 2) (16596,)
Percentage: 19.33%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Test:
(17864, 2) (17864,)
Percentage: 20.81%
Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%

Total samples: 85846
Ratio (Train:Validation:Test): 51386:16596:17864
Overall Label Ratio: Label 0.0: 50.00%, Label 1.0: 50.00%


In [14]:
drug_response = drug_response.fillna(0)
drug_response

,22Rv1,23132/87,253J,253J-BV,42-MG-BA,5637,639-V,647-V,697,769-P,...,YAPC,YD-10B,YD-15,YD-38,YD-8,YH-13,YKG1,ZR-75-1,ZR-75-30,huH-1
3-Cl-AHPC,1.148373,1.038346,0.666293,1.288448,1.497660,1.060512,1.851229,1.219439,0.934974,1.949833,...,0.913961,0.771345,1.149942,0.509339,0.074496,0.485174,-0.227924,-0.247160,0.774958,0.887548
A-804598,0.000000,0.000000,0.000000,0.000000,-0.631499,0.000000,0.000000,-0.922372,0.000000,0.000000,...,-0.630289,0.000000,0.000000,0.000000,-0.753520,-0.707025,0.000000,-0.626612,-0.371135,-0.649507
ABT-199,0.000000,0.000000,0.000000,0.000000,-1.049029,0.000000,0.000000,-0.855483,1.928232,0.000000,...,-1.033353,0.000000,0.000000,0.000000,-0.749157,-1.120671,-0.802896,-0.626612,-0.673942,-0.649507
ABT-737,-0.059864,-0.610808,-0.182238,0.037168,-0.970819,-0.228234,-0.851591,0.000000,0.908925,0.018693,...,-0.630289,-0.746309,-0.714201,-0.636043,0.000000,0.000000,-0.653919,-0.353114,-0.535715,0.000000
AC55649,-0.452787,-0.279801,-0.709693,-0.429878,0.000000,-0.441566,-0.823075,-0.810171,-1.094459,-0.652714,...,-0.630289,-0.471195,-1.111169,-0.712088,-0.557666,-0.620566,-0.596471,-0.817642,-0.712228,-1.137469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,-0.395635,-0.758664,-0.218130,-0.505584,-0.850652,-0.618960,-0.500372,0.647719,-0.494992,-0.879207,...,-1.337448,-1.034045,-0.569849,-0.585477,-0.371508,0.093990,-0.567260,-0.626612,0.000000,1.175953
vincristine,3.522921,3.225466,1.913537,1.962811,1.785043,2.141431,1.795148,2.579225,2.723607,1.564796,...,1.149497,1.951937,2.353822,2.322626,1.724566,3.295211,3.148098,1.424617,1.414880,1.671517
vorapaxar,-0.218373,-0.538253,-0.408513,-0.272643,-0.288921,-0.121186,-1.074965,-0.797585,0.000000,-0.534172,...,-0.546801,0.089064,-0.408000,-0.632123,0.647999,-0.643876,0.000000,-0.141728,-0.666484,-0.359676
vorinostat,1.077379,0.559876,0.722082,0.465002,0.113130,0.505046,0.039052,0.381242,0.089795,0.616956,...,0.238870,0.000000,0.185810,3.210082,0.297013,-0.027222,0.303718,0.247118,0.484084,0.525971


# Masking

In [15]:
# Validation Mask
for _, row in X_val.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# Test Mask
for _, row in X_test.iterrows():
    drug_response.loc[row["Drug"], row["Cell"]] = 0

# DTI

In [16]:
dti = pd.read_csv("../data/full_table.csv")
dti = dti.dropna(subset="Drug Name").reset_index(drop=True)
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
0,Lepirudin,DB00001,NaN,46507011.0,NaN,F2,NaN
1,Cetuximab,DB00002,NaN,46507042.0,NaN,EGFR,NaN
2,Cetuximab,DB00002,NaN,46507042.0,NaN,FCGR3B,NaN
3,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QA,NaN
4,Cetuximab,DB00002,NaN,46507042.0,NaN,C1QB,NaN


In [17]:
dti = dti[dti["Drug Name"].isin(drug_response.index)]
dti.head()

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
11264,XL765,DB05241,NaN,347910044.0,NaN,PIK3CA,NaN
11265,XL765,DB05241,NaN,347910044.0,NaN,PIK3CG,NaN
11266,XL765,DB05241,NaN,347910044.0,NaN,PIK3CD,NaN
11267,XL765,DB05241,NaN,347910044.0,NaN,PIK3CB,NaN
11268,XL765,DB05241,NaN,347910044.0,NaN,MTOR,NaN


In [18]:
print("unique drugs:", len(set(dti["Drug Name"])))
print("unique genes:", len(set(dti.Gene)))

unique drugs: 10
unique genes: 17


In [19]:
len(set(drug_response.index) & set(dti["Drug Name"]))

10

In [20]:
len(set(gene_exp.columns) & set(dti.Gene))

17

## All drugs are in drug response.

In [21]:
dti = dti[dti.Gene.isin(set(gene_exp.columns) & set(dti.Gene))]
dti

,Drug Name,DrugBank ID,PubChem CID,PubChem SID,SMILES,Gene,NSC
11264,XL765,DB05241,NaN,347910044.0,NaN,PIK3CA,NaN
11265,XL765,DB05241,NaN,347910044.0,NaN,PIK3CG,NaN
11266,XL765,DB05241,NaN,347910044.0,NaN,PIK3CD,NaN
11267,XL765,DB05241,NaN,347910044.0,NaN,PIK3CB,NaN
11268,XL765,DB05241,NaN,347910044.0,NaN,MTOR,NaN
11414,PX-12,DB05448,219104.0,175427007.0,CCC(C)SSC1=NC=CN1,TXNRD1,NaN
11628,OSI-930,DB05913,9868037.0,347827749.0,FC(F)(F)OC1=CC=C(NC(=O)C2=C(NCC3=CC=NC4=CC=CC=...,KIT,NaN
11629,OSI-930,DB05913,9868037.0,347827749.0,FC(F)(F)OC1=CC=C(NC(=O)C2=C(NCC3=CC=NC4=CC=CC=...,FLT1,NaN
11662,SNS-032,DB05969,NaN,NaN,CC(C)(C)C1=CN=C(CSC2=CN=C(NC(=O)C3CCNCC3)S2)O1,CDK7,767048.0
11663,SNS-032,DB05969,NaN,NaN,CC(C)(C)C1=CN=C(CSC2=CN=C(NC(=O)C3CCNCC3)S2)O1,CDK7,799362.0


In [22]:
dti.shape

(30, 7)

# Selected highly variable genes

In [23]:
variance = gene_exp.std()
variance = variance.sort_values(ascending=False)
variance = pd.DataFrame(variance > np.percentile(variance, 90))
variance = list(variance[variance[0]].index)
len(variance)

1985

In [24]:
print("DTI unique genes: ", len(set(dti["Gene"])))
print("Top 90% variable genes: ", len(variance))
print("Total: ", len(set(variance) | (set(dti["Gene"]))))

DTI unique genes:  17
Top 90% variable genes:  1985
Total:  1999


# Preprocessed data dims

In [25]:
genes = sorted(set(variance) | set(dti["Gene"]))
gene_exp = gene_exp[genes]
gene_exp.shape

(823, 1999)

# Normalize

In [26]:
gene_norm_cell = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp),
    index=gene_exp.index,
    columns=gene_exp.columns,
)
gene_norm_cell

,AASS,ABAT,ABCA12,ABCB1,ABCC2,ABCG1,ABCG2,ABHD17C,ABLIM1,ABLIM3,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
22Rv1,0.079485,0.116840,-0.483094,-0.552779,0.456874,-0.695633,-0.027558,0.129877,0.182127,-0.143379,...,0.811417,-0.442241,-0.569299,-0.488509,0.416455,0.776187,0.117425,0.638045,-0.545090,-0.873126
23132/87,-0.190854,-0.716881,3.107916,-0.515210,2.177959,3.122479,-0.784309,1.844905,2.293456,-1.031825,...,-0.872435,-0.423262,-0.886693,-0.778370,0.065100,-0.857099,-0.978051,-0.823943,-1.313457,-1.266730
253J,0.081382,0.117624,1.392111,2.102455,-0.137017,-0.285471,-0.775249,-0.602398,0.791137,-0.845161,...,-1.089384,-0.379853,-0.865039,-0.689803,-0.840133,0.992254,-0.471747,1.556026,0.672945,-0.678637
253J-BV,-0.096571,-0.422280,1.919913,-0.044270,-0.219260,-0.463487,-0.456652,-0.189421,0.618289,-0.380353,...,-1.099414,-0.355236,-0.699368,-0.647410,-0.826878,1.060980,-0.801150,1.175106,0.310896,0.354670
42-MG-BA,0.646769,1.218113,-0.326291,-0.529339,-0.368735,-0.749457,0.037933,-0.772089,-1.599759,0.863911,...,0.855164,-0.434015,1.490415,0.922669,0.757065,0.470108,0.398356,1.711350,0.627044,-0.918296
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YH-13,1.204372,-0.862517,-0.632155,-0.375033,-0.699409,-0.724897,-0.876699,-0.119432,-0.513373,0.958667,...,-0.105397,-0.388259,0.385024,0.415019,0.669169,0.135917,1.492647,1.507893,1.482460,-0.907781
YKG1,1.829917,-0.363433,-0.512386,-0.552734,-0.217558,0.000710,0.995143,-0.821718,-0.583702,0.100419,...,0.657551,-0.530904,0.426417,1.256510,0.420141,0.935498,-0.383598,1.982649,1.054158,-0.411265
ZR-75-1,-1.034452,1.963944,3.179435,-0.453431,0.694087,1.271752,-0.103097,-0.149091,0.503679,-1.016187,...,1.032501,1.137550,-0.883569,-0.635667,0.167622,-0.855169,-0.812868,0.091936,-0.590477,0.959040
ZR-75-30,-1.226454,2.216909,-0.587684,-0.499693,1.690709,2.281731,-0.913486,1.478618,0.932523,0.483595,...,-1.114061,2.449396,-0.649491,-0.886172,-0.005162,-0.117277,-1.142303,-0.020917,-1.250117,0.871342


In [27]:
gene_norm_gene = pd.DataFrame(
    StandardScaler().fit_transform(gene_exp.T),
    index=gene_exp.columns,
    columns=gene_exp.index,
).T
gene_norm_gene

,AASS,ABAT,ABCA12,ABCB1,ABCC2,ABCG1,ABCG2,ABHD17C,ABLIM1,ABLIM3,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
22Rv1,-0.368959,-0.116520,-1.295561,-1.055908,-0.104458,-0.862828,0.009211,1.063436,0.889897,-0.089153,...,0.493045,-1.460895,-0.636189,-0.898323,0.723836,0.437672,-0.465764,-0.013862,-0.442247,-0.690521
23132/87,-0.497298,-0.605211,0.715117,-0.938347,1.072029,1.254569,-0.554050,1.833025,1.955896,-0.667575,...,-0.947850,-1.307988,-0.775255,-1.026131,0.395555,-0.561400,-1.033643,-1.166389,-0.891245,-0.866088
253J,-0.424719,-0.195678,-0.257617,0.499694,-0.605738,-0.649876,-0.627880,0.477114,1.071563,-0.629160,...,-1.218455,-1.382461,-0.848287,-1.054175,-0.249392,0.433670,-0.843338,0.625557,0.281101,-0.601521
253J-BV,-0.588974,-0.578285,-0.014549,-0.828830,-0.735193,-0.822405,-0.454834,0.690311,0.970706,-0.382548,...,-1.322183,-1.469494,-0.822592,-1.109905,-0.293785,0.451342,-1.114405,0.294574,0.010705,-0.022990
42-MG-BA,-0.158291,0.397896,-1.221019,-1.075439,-0.811134,-0.944445,-0.083076,0.330794,-0.357308,0.431332,...,0.340942,-1.448682,0.525041,0.071397,0.732028,0.079065,-0.402426,0.691305,0.199760,-0.786972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YH-13,0.049932,-0.903756,-1.468528,-1.074429,-1.130401,-1.019758,-0.834430,0.579636,0.164275,0.390346,...,-0.540644,-1.506108,-0.234609,-0.389698,0.571549,-0.209662,0.103639,0.429078,0.632748,-0.870804
YKG1,0.465432,-0.597576,-1.430896,-1.189018,-0.787789,-0.611939,0.585937,0.267394,0.183813,-0.114773,...,0.136480,-1.623965,-0.169198,0.276167,0.488172,0.312268,-0.923576,0.895048,0.439031,-0.558371
ZR-75-1,-1.033443,1.068149,0.867174,-0.975664,0.082575,0.323556,-0.047344,0.887505,1.083342,-0.702386,...,0.688939,-0.432454,-0.831236,-0.999549,0.542074,-0.594076,-1.020632,-0.480292,-0.464277,0.515322
ZR-75-30,-1.060646,1.179911,-1.236630,-0.924622,0.820637,0.892958,-0.627498,1.766275,1.295440,0.360145,...,-1.148622,0.401113,-0.615921,-1.106234,0.417449,-0.099158,-1.127606,-0.521978,-0.843415,0.453332


# Make matrices association matrices by setting 0 threshold and min max scaler.

In [28]:
def min_max_scale(data):
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = data[data > 0].fillna(0)
    return pd.DataFrame(
        scaler.fit_transform(data), index=data.index, columns=data.columns
    )

In [29]:
A_dc = min_max_scale(drug_response)
A_dc

,22Rv1,23132/87,253J,253J-BV,42-MG-BA,5637,639-V,647-V,697,769-P,...,YAPC,YD-10B,YD-15,YD-38,YD-8,YH-13,YKG1,ZR-75-1,ZR-75-30,huH-1
3-Cl-AHPC,0.252292,0.269449,0.149900,0.301313,0.405731,0.247627,0.394261,0.314577,0.342493,0.000000,...,0.141552,0.000000,0.000000,0.108917,0.018171,0.000000,0.000000,0.000000,0.000000,0.000000
A-804598,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABT-199,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.706336,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
ABT-737,0.000000,0.000000,0.000000,0.008692,0.000000,0.000000,0.000000,0.000000,0.332951,0.005748,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
AC55649,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.022763,0.000000,0.000000,0.000000,0.369900
vincristine,0.773970,0.837002,0.430499,0.459018,0.483586,0.500019,0.382317,0.665360,0.997693,0.000000,...,0.178031,0.000000,0.000000,0.496672,0.420653,0.000000,0.000000,0.330205,0.000000,0.000000
vorapaxar,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.032608,0.000000,0.000000,0.158059,0.000000,0.000000,0.000000,0.000000,0.000000
vorinostat,0.236695,0.145287,0.162451,0.108744,0.030648,0.117927,0.008317,0.098349,0.032893,0.189699,...,0.036996,0.000000,0.061141,0.686445,0.072447,0.000000,0.095432,0.057278,0.108013,0.165446


In [30]:
A_cg = min_max_scale(gene_norm_gene + gene_norm_cell)
A_cg

,AASS,ABAT,ABCA12,ABCB1,ABCC2,ABCG1,ABCG2,ABHD17C,ABLIM1,ABLIM3,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
22Rv1,0.000000,0.000060,0.000000,0.000000,0.059533,0.000000,0.000000,0.274096,0.219203,0.000000,...,0.310612,0.000000,0.000000,0.000000,0.333559,0.332779,0.000000,0.126988,0.000000,0.000000
23132/87,0.000000,0.000000,0.647699,0.000000,0.549014,0.788903,0.000000,0.844794,0.868891,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.134751,0.000000,0.000000,0.000000,0.000000,0.000000
253J,0.000000,0.000000,0.192206,0.406358,0.000000,0.000000,0.000000,0.000000,0.380878,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.390917,0.000000,0.443837,0.271512,0.000000
253J-BV,0.000000,0.000000,0.322807,0.000000,0.000000,0.000000,0.000000,0.115051,0.324911,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.414603,0.000000,0.299002,0.091524,0.089927
42-MG-BA,0.115638,0.302933,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.287772,...,0.284811,0.000000,0.431530,0.193489,0.435591,0.150556,0.000000,0.488814,0.235300,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YH-13,0.296933,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.105705,0.000000,0.299718,...,0.000000,0.000000,0.032205,0.004929,0.362936,0.000000,0.416933,0.394071,0.601967,0.000000
YKG1,0.543380,0.000000,0.000000,0.000000,0.000000,0.000000,0.242771,0.000000,0.000000,0.000000,...,0.189071,0.000000,0.055073,0.298327,0.265701,0.342075,0.000000,0.585459,0.424947,0.000000
ZR-75-1,0.000000,0.568389,0.685578,0.000000,0.131200,0.287532,0.000000,0.169608,0.324508,0.000000,...,0.409901,0.111397,0.000000,0.000000,0.207601,0.000000,0.000000,0.000000,0.000000,0.399739
ZR-75-30,0.000000,0.636760,0.000000,0.000000,0.424236,0.572194,0.000000,0.745329,0.455565,0.187458,...,0.000000,0.450347,0.000000,0.000000,0.120603,0.000000,0.000000,0.000000,0.000000,0.359154


In [31]:
A_dg = (
    pd.DataFrame(
        np.ones([len(A_dc.index), len(A_cg.columns)]),
        index=A_dc.index,
        columns=A_cg.columns,
    )
    / 2
)
for _, i in dti.iterrows():
    A_dg.loc[i["Drug Name"], i["Gene"]] = 1
A_dg

,AASS,ABAT,ABCA12,ABCB1,ABCC2,ABCG1,ABCG2,ABHD17C,ABLIM1,ABLIM3,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
3-Cl-AHPC,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
A-804598,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ABT-199,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
ABT-737,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
AC55649,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
vincristine,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
vorapaxar,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5
vorinostat,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,...,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5,0.5


In [32]:
drug_response

,22Rv1,23132/87,253J,253J-BV,42-MG-BA,5637,639-V,647-V,697,769-P,...,YAPC,YD-10B,YD-15,YD-38,YD-8,YH-13,YKG1,ZR-75-1,ZR-75-30,huH-1
3-Cl-AHPC,1.148373,1.038346,0.666293,1.288448,1.497660,1.060512,1.851229,1.219439,0.934974,0.000000,...,0.913961,0.000000,0.000000,0.509339,0.074496,0.000000,0.000000,-0.247160,0.000000,0.000000
A-804598,0.000000,0.000000,0.000000,0.000000,-0.631499,0.000000,0.000000,-0.922372,0.000000,0.000000,...,-0.630289,0.000000,0.000000,0.000000,-0.753520,-0.707025,0.000000,-0.626612,-0.371135,-0.649507
ABT-199,0.000000,0.000000,0.000000,0.000000,-1.049029,0.000000,0.000000,-0.855483,1.928232,0.000000,...,-1.033353,0.000000,0.000000,0.000000,-0.749157,-1.120671,0.000000,-0.626612,0.000000,0.000000
ABT-737,-0.059864,-0.610808,-0.182238,0.037168,-0.970819,-0.228234,-0.851591,0.000000,0.908925,0.018693,...,-0.630289,-0.746309,-0.714201,-0.636043,0.000000,0.000000,-0.653919,-0.353114,-0.535715,0.000000
AC55649,0.000000,-0.279801,0.000000,0.000000,0.000000,-0.441566,0.000000,-0.810171,-1.094459,-0.652714,...,-0.630289,-0.471195,-1.111169,-0.712088,-0.557666,-0.620566,-0.596471,-0.817642,-0.712228,-1.137469
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,0.000000,-0.758664,-0.218130,-0.505584,-0.850652,0.000000,0.000000,0.000000,-0.494992,-0.879207,...,0.000000,-1.034045,-0.569849,-0.585477,-0.371508,0.093990,-0.567260,-0.626612,0.000000,1.175953
vincristine,3.522921,3.225466,1.913537,1.962811,1.785043,2.141431,1.795148,2.579225,2.723607,0.000000,...,1.149497,0.000000,0.000000,2.322626,1.724566,0.000000,0.000000,1.424617,0.000000,0.000000
vorapaxar,-0.218373,-0.538253,-0.408513,-0.272643,-0.288921,-0.121186,-1.074965,-0.797585,0.000000,-0.534172,...,-0.546801,0.089064,-0.408000,-0.632123,0.647999,-0.643876,0.000000,-0.141728,-0.666484,-0.359676
vorinostat,1.077379,0.559876,0.722082,0.465002,0.113130,0.505046,0.039052,0.381242,0.089795,0.616956,...,0.238870,0.000000,0.185810,3.210082,0.297013,-0.027222,0.303718,0.247118,0.484084,0.525971


In [33]:
print(
    "Drug Density: ",
    round(len(A_dc.values.nonzero()[0]) / drug_response.size, 4) * 100,
    "%",
)
print("Cell Density: ", round(len(A_cg.values.nonzero()[0]) / A_cg.size, 4) * 100, "%")
print("Gene Density: ", round(len(A_dg.values.nonzero()[0]) / A_dg.size, 5) * 100, "%")

Drug Density:  26.99 %
Cell Density:  44.28 %
Gene Density:  100.0 %


# Similarity

In [34]:
def normalize_similarity_matrix(df, gamma=None):
    similarity_matrix = rbf_kernel(df.values, gamma=gamma)
    scaler = MinMaxScaler()
    normalized_matrix = scaler.fit_transform(similarity_matrix.reshape(-1, 1))
    normalized_df = pd.DataFrame(
        normalized_matrix.reshape(similarity_matrix.shape),
        index=df.index,
        columns=df.index,
    )

    return normalized_df

In [35]:
cell_sim = normalize_similarity_matrix(drug_response.T)
cell_sim.to_csv("../CTRP_data/cell_sim.csv")
cell_sim

,22Rv1,23132/87,253J,253J-BV,42-MG-BA,5637,639-V,647-V,697,769-P,...,YAPC,YD-10B,YD-15,YD-38,YD-8,YH-13,YKG1,ZR-75-1,ZR-75-30,huH-1
22Rv1,1.000000,0.592607,0.685965,0.469995,0.281459,0.668219,0.544736,0.552746,0.381270,0.400914,...,0.497554,0.355656,0.384390,0.545434,0.352756,0.341054,0.311906,0.327655,0.229838,0.286065
23132/87,0.592607,1.000000,0.520028,0.378138,0.349874,0.467794,0.487573,0.423789,0.408057,0.470648,...,0.405768,0.426549,0.455018,0.559484,0.353540,0.391371,0.358346,0.360125,0.203712,0.343371
253J,0.685965,0.520028,1.000000,0.554627,0.314715,0.640042,0.566608,0.496580,0.381908,0.428245,...,0.490055,0.394956,0.413266,0.515705,0.375662,0.388491,0.341867,0.279179,0.273487,0.319504
253J-BV,0.469995,0.378138,0.554627,1.000000,0.209296,0.425470,0.512291,0.395573,0.323140,0.395233,...,0.450915,0.355866,0.357149,0.400393,0.247430,0.343590,0.315124,0.248244,0.251073,0.291853
42-MG-BA,0.281459,0.349874,0.314715,0.209296,1.000000,0.249266,0.285903,0.365651,0.419183,0.361648,...,0.321605,0.319288,0.359288,0.323350,0.380990,0.387820,0.421632,0.257317,0.252893,0.367887
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
YH-13,0.341054,0.391371,0.388491,0.343590,0.387820,0.399898,0.339251,0.479600,0.458709,0.568053,...,0.370386,0.521184,0.583273,0.348838,0.406279,1.000000,0.721737,0.309727,0.453525,0.647003
YKG1,0.311906,0.358346,0.341867,0.315124,0.421632,0.317596,0.332877,0.398685,0.509950,0.557730,...,0.334249,0.496753,0.535085,0.332030,0.374115,0.721737,1.000000,0.324863,0.441504,0.605022
ZR-75-1,0.327655,0.360125,0.279179,0.248244,0.257317,0.260873,0.238171,0.286636,0.282314,0.330731,...,0.436058,0.297233,0.293527,0.361631,0.346170,0.309727,0.324863,1.000000,0.250648,0.282970
ZR-75-30,0.229838,0.203712,0.273487,0.251073,0.252893,0.258820,0.177798,0.241953,0.272566,0.405690,...,0.248963,0.366463,0.381374,0.259043,0.255314,0.453525,0.441504,0.250648,1.000000,0.401213


In [36]:
print("Min:", np.min(cell_sim.values))
print("Max:", np.max(cell_sim.values))
print("Mean:", np.mean(cell_sim.values))
print("Median:", np.median(cell_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.38302626429805464
Median: 0.369475061999495


In [37]:
gene_sim = normalize_similarity_matrix(gene_norm_cell.T)
gene_sim.to_csv("../nci_data/gene_sim.csv")
gene_sim

,AASS,ABAT,ABCA12,ABCB1,ABCC2,ABCG1,ABCG2,ABHD17C,ABLIM1,ABLIM3,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
AASS,1.000000,0.103080,0.070450,0.110127,0.090754,0.076116,0.115020,0.113161,0.061278,0.138480,...,0.135466,0.080942,0.183031,0.144649,0.126224,0.134135,0.154953,0.184583,0.140081,0.098145
ABAT,0.103080,1.000000,0.183348,0.130557,0.080864,0.197354,0.094468,0.126948,0.140651,0.078571,...,0.158332,0.176080,0.107497,0.105755,0.131447,0.134892,0.118328,0.113287,0.130770,0.100662
ABCA12,0.070450,0.183348,1.000000,0.103499,0.144614,0.184260,0.136951,0.187107,0.208169,0.158160,...,0.087632,0.330285,0.056428,0.067642,0.105069,0.079614,0.081609,0.091581,0.094277,0.119495
ABCB1,0.110127,0.130557,0.103499,1.000000,0.158863,0.102901,0.102390,0.105142,0.152433,0.096527,...,0.151214,0.079846,0.085015,0.132549,0.080968,0.115814,0.092787,0.095107,0.094114,0.090660
ABCC2,0.090754,0.080864,0.144614,0.158863,1.000000,0.110997,0.232710,0.162490,0.162628,0.146133,...,0.079385,0.118206,0.064714,0.078416,0.076056,0.088402,0.093362,0.104782,0.077090,0.111452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF850,0.134135,0.134892,0.079614,0.115814,0.088402,0.090326,0.077184,0.057169,0.062820,0.068897,...,0.206159,0.071777,0.246303,0.150164,0.288078,1.000000,0.295145,0.175089,0.314336,0.141582
ZNF880,0.154953,0.118328,0.081609,0.092787,0.093362,0.081586,0.106198,0.068621,0.072782,0.126673,...,0.132727,0.085131,0.220492,0.137986,0.271997,0.295145,1.000000,0.226676,0.353348,0.129161
ZNF883,0.184583,0.113287,0.091581,0.095107,0.104782,0.092520,0.130744,0.107680,0.088818,0.157830,...,0.132764,0.101238,0.167526,0.125684,0.176204,0.175089,0.226676,1.000000,0.233058,0.148129
ZSCAN18,0.140081,0.130770,0.094277,0.094114,0.077090,0.112951,0.082747,0.057385,0.089272,0.101783,...,0.201759,0.092279,0.250528,0.131756,0.295912,0.314336,0.353348,0.233058,1.000000,0.190847


In [38]:
print("Min:", np.min(gene_sim.values))
print("Max:", np.max(gene_sim.values))
print("Mean:", np.mean(gene_sim.values))
print("Median:", np.median(gene_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.13722283587617576
Median: 0.12200924140060722


# NSC to SMILES

In [39]:
def get_morgan_fingerprint(SMILES):
    # Initialize parser parameters
    params = Chem.SmilesParserParams()
    params.useChirality = True  # Preserve stereochemistry information
    params.removeHs = False  # Keep hydrogen atoms
    mfp = []

    for smi in SMILES:
        mol = None
        # Sanitization attempt strategies
        sanitize_attempts = [
            {"sanitize": True},  # First try with standard sanitization
            {
                "sanitize": False,
                "partial_sanitize": True,
            },  # Fallback: partial sanitization
        ]

        for attempt in sanitize_attempts:
            try:
                # Update parameters for this attempt
                current_params = Chem.SmilesParserParams()
                current_params.sanitize = attempt["sanitize"]
                current_params.useChirality = params.useChirality
                current_params.removeHs = params.removeHs

                # Molecule object creation
                mol = Chem.MolFromSmiles(smi, params=current_params)

                if mol and "partial_sanitize" in attempt:
                    # Perform partial sanitization (skip property validation)
                    Chem.SanitizeMol(mol, Chem.SANITIZE_ALL ^ Chem.SANITIZE_PROPERTIES)

                if mol:  # Successfully processed molecule
                    # Generate Morgan fingerprint
                    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
                    mfp.append(np.array(fp))
                    break  # Exit attempt loop on success

            except Exception as e:
                if attempt == sanitize_attempts[-1]:  # Final attempt failed
                    print(f"Processing failed: {smi}")
                    print(f"Error details: {str(e)}")
                continue  # Try next attempt

        if not mol:  # All attempts failed
            print(f"Complete processing failure: {smi}")
            mfp.append(np.zeros(2048))  # Insert zero-vector placeholder

    return np.array(mfp)

In [40]:
conv = dict(SMILES[["drugs", "SMILES"]].values)
mfp = get_morgan_fingerprint([conv[i] for i in drug_response.index])
mfp = pd.DataFrame(mfp, index=drug_response.index)

In [41]:
drug_sim = normalize_similarity_matrix(mfp)
drug_sim.to_csv("../CTRP_data/drug_sim.csv")
drug_sim

,3-Cl-AHPC,A-804598,ABT-199,ABT-737,AC55649,AGK-2,AM-580,AT-406,AT13387,AT7867,...,triptolide,tubastatin A,valdecoxib,vandetanib,veliparib,vemurafenib,vincristine,vorapaxar,vorinostat,zebularine
3-Cl-AHPC,1.000000,0.596617,0.412804,0.475304,0.709540,0.591736,0.719420,0.552773,0.635751,0.709540,...,0.596617,0.611274,0.679960,0.543057,0.645558,0.581981,0.431992,0.547914,0.709540,0.645558
A-804598,0.596617,1.000000,0.369766,0.480128,0.694739,0.754074,0.684884,0.596617,0.601500,0.645558,...,0.581981,0.635751,0.684884,0.596617,0.650466,0.596617,0.398437,0.533349,0.724363,0.679960
ABT-199,0.412804,0.369766,1.000000,0.547914,0.431992,0.374539,0.451218,0.317421,0.417597,0.470482,...,0.360228,0.422393,0.441600,0.403223,0.417597,0.489783,0.227666,0.350699,0.422393,0.436795
ABT-737,0.475304,0.480128,0.547914,1.000000,0.552773,0.436795,0.533349,0.446408,0.499448,0.504284,...,0.403223,0.552773,0.533349,0.417597,0.460845,0.504284,0.270078,0.355462,0.552773,0.489783
AC55649,0.709540,0.694739,0.431992,0.552773,1.000000,0.621058,0.808771,0.640653,0.714479,0.739208,...,0.645558,0.739208,0.759035,0.630851,0.724363,0.689810,0.480128,0.606386,0.818748,0.734257
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
vemurafenib,0.581981,0.596617,0.489783,0.504284,0.689810,0.572236,0.621058,0.494614,0.567366,0.679960,...,0.509122,0.581981,0.630851,0.581981,0.596617,1.000000,0.355462,0.547914,0.650466,0.625953
vincristine,0.431992,0.398437,0.227666,0.270078,0.480128,0.345938,0.470482,0.384091,0.417597,0.422393,...,0.379314,0.470482,0.441600,0.403223,0.494614,0.355462,1.000000,0.331669,0.480128,0.446408
vorapaxar,0.547914,0.533349,0.350699,0.355462,0.606386,0.499448,0.567366,0.489783,0.504284,0.557635,...,0.494614,0.528499,0.596617,0.509122,0.543057,0.547914,0.331669,1.000000,0.596617,0.611274
vorinostat,0.709540,0.724363,0.422393,0.552773,0.818748,0.660287,0.798804,0.670119,0.694739,0.719420,...,0.645558,0.749116,0.759035,0.640653,0.724363,0.650466,0.480128,0.596617,1.000000,0.754074


In [42]:
print("Min:", np.min(drug_sim.values))
print("Max:", np.max(drug_sim.values))
print("Mean:", np.mean(drug_sim.values))
print("Median:", np.median(drug_sim.values))

Min: 0.0
Max: 1.0
Mean: 0.5762820514564007
Median: 0.581980933476709


# Unified Graph

In [43]:
indexes = list(A_dc.index) + list(A_cg.index) + list(A_dg.columns)
n_all = len(indexes)
base = pd.DataFrame(np.zeros([n_all, n_all]), index=indexes, columns=indexes)
base

,3-Cl-AHPC,A-804598,ABT-199,ABT-737,AC55649,AGK-2,AM-580,AT-406,AT13387,AT7867,...,ZNF711,ZNF750,ZNF788,ZNF804A,ZNF83,ZNF850,ZNF880,ZNF883,ZSCAN18,ZYG11A
3-Cl-AHPC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
A-804598,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABT-199,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ABT-737,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
AC55649,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZNF850,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF880,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZNF883,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
ZSCAN18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [44]:
base.loc[A_cg.index, A_cg.columns] = A_cg
base.loc[A_cg.columns, A_cg.index] = A_cg.T
base.loc[A_dc.index, A_dc.columns] = A_dc
base.loc[A_dc.columns, A_dc.index] = A_dc.T
base.loc[A_dg.index, A_dg.columns] = A_dg
base.loc[A_dg.columns, A_dg.index] = A_dg.T

In [45]:
edge_index = np.array(base.values.nonzero())
edge_attr = np.array(base.values[base.values.nonzero()])

np.save(
    "../CTRP_data/idxs.npy",
    pd.DataFrame([list(range(len(base.index))), base.index]).values,
)

np.save(
    "../CTRP_data/edge_idxs.npy",
    edge_index,
)

np.save(
    "../CTRP_data/edge_attr.npy",
    edge_attr,
)